In [14]:
#pip install numpy rasterio pytecplot

In [15]:
import tifffile as tiff
import numpy as np

data = tiff.memmap("../../ibcao_v5_1_2025_depth_400m.tif")  # does NOT load full file
ny, nx = data.shape

print(nx, ny)

14550 14550


In [16]:
# Downsampling factor
factor = 1

data_small = data[::factor, ::factor]

ny, nx = data_small.shape
print(f"New size: {nx} x {ny}")

with open("bathymetry.dat", "w") as f:
    f.write('TITLE = "Arctic Bathymetry"\n')
    f.write('VARIABLES = "X", "Y", "Depth"\n')
    f.write(f'ZONE I={nx}, J={ny}, F=POINT\n')

    for j in range(ny):
        for i in range(nx):
            f.write(f"{i} {j} {data_small[j, i]}\n")

print("Done!")

New size: 14550 x 14550


KeyboardInterrupt: 

# Convert netcdf to tecplot format  

In [16]:
import tecio
from netCDF4 import Dataset
import numpy as np

nc = Dataset("../../gmm_clusters.nc")

In [17]:
nc.variables.keys()

dict_keys(['Latitude_[deg_N]', 'Longitude_[deg_E]', 'Depth_[m]', 'gmm_label', 'prob_cluster_0', 'prob_cluster_1', 'prob_cluster_2', 'prob_cluster_3', 'prob_cluster_4', 'prob_cluster_5', 'prob_cluster_6', 'prob_cluster_7', 'prob_cluster_8', 'index'])

In [27]:
lon = nc.variables['Longitude_[deg_E]'][:]
lat = nc.variables['Latitude_[deg_N]'][:]
depth = nc.variables['Depth_[m]'][:]
label = nc.variables['gmm_label'][:]
probs = [nc.variables[f'prob_cluster_{i}'][:] for i in range(9)]
N = len(depth)

In [26]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)

x, y = transformer.transform(lon, lat)

In [29]:
with open("gmm_animation.dat", "w") as f:
    f.write('TITLE = "GMM Animation"\n')
    f.write('VARIABLES = "X", "Y", "Depth", "Prob"\n')

    for k in range(9):
        f.write(f'ZONE T="Prob_{k}", I={N}, F=POINT\n')

        for i in range(N):
            f.write(f"{x[i]} {y[i]} {depth[i]} {probs[k][i]}\n")

print("Ready for animation in Tecplot")

Ready for animation in Tecplot


# Retry

In [1]:
from netCDF4 import Dataset
import numpy as np
from pyproj import Transformer

In [2]:
nc = Dataset("../../gmm_clusters.nc")

In [4]:
# Load variables
lon = nc.variables['Longitude_[deg_E]'][:]
lat = nc.variables['Latitude_[deg_N]'][:]
depth = nc.variables['Depth_[m]'][:]
label = nc.variables['gmm_label'][:]

# Convert to polar stereographic projection (EPSG:3413)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)
x, y = transformer.transform(lon, lat)

N = len(depth)

# Write tecplot file with x, y, depth and gmm labels only
with open("../../tecplot_ready.dat", "w") as f:
    f.write('TITLE = "GMM Bathymetry"\n')
    f.write('VARIABLES = "X", "Y", "Depth", "Label"\n')

    # Single clean zone
    f.write(f'ZONE T="Data", I={N}, F=POINT\n')

    for i in range(N):
        f.write(f"{x[i]} {y[i]} {depth[i]} {label[i]}\n")

print("File ready for Tecplot")

File ready for Tecplot


In [5]:
print(x[:5])

[2077594.20246383 2079540.34920533 2076806.95475523 2078332.54891726
 2075388.03754901]


# Reretry

In [17]:
from netCDF4 import Dataset
import numpy as np
from pyproj import Transformer
from scipy.interpolate import griddata

In [18]:
# Load data
nc = Dataset("../../gmm_clusters.nc")

In [19]:
lon = nc.variables['Longitude_[deg_E]'][:]
lat = nc.variables['Latitude_[deg_N]'][:]
depth = nc.variables['Depth_[m]'][:]
label = nc.variables['gmm_label'][:]

# Convert to polar stereographic projection
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)
x, y = transformer.transform(lon, lat)

In [20]:
# Downsample (IMPORTANT for memory)
step = 3
x = x[::step]
y = y[::step]
depth = depth[::step]
label = label[::step]

In [21]:
# Create structured 2D grid
nx, ny = 200, 200  # adjust resolution
xi = np.linspace(x.min(), x.max(), nx)
yi = np.linspace(y.min(), y.max(), ny)

XI, YI = np.meshgrid(xi, yi)

In [22]:
# Interpolate surface
depth_grid = griddata((x, y), depth, (XI, YI), method='linear')
label_grid = griddata((x, y), label, (XI, YI), method='nearest')

In [23]:
# Create vertical levels
nz = 50
z_levels = np.linspace(np.nanmin(depth_grid), 0, nz)  # seabed → surface

In [27]:
# Build 3D volume
# Shape: (K, J, I)
Depth3D = np.zeros((nz, ny, nx))
Label3D = np.zeros((nz, ny, nx))

for k in range(nz):
    z = z_levels[k]

    for j in range(ny):
        for i in range(nx):
            # Always assign values (no gaps)
            Depth3D[k, j, i] = z
            Label3D[k, j, i] = label_grid[j, i]

# ---- Write Tecplot file ----
with open("../../3D_volume.dat", "w") as f:
    f.write('TITLE = "3D Bathymetry Volume"\n')
    f.write('VARIABLES = "X", "Y", "Z", "Label"\n')
    f.write(f'ZONE I={nx}, J={ny}, K={nz}, DATAPACKING=POINT\n')
    f.write(f'VARLOCATION=([4]=NODAL)\n')

    for k in range(nz):
        for j in range(ny):
            for i in range(nx):
                f.write(f"{XI[j,i]} {YI[j,i]} {Depth3D[k,j,i]} {Label3D[k,j,i]}\n")

                

print("3D Tecplot file created!")

3D Tecplot file created!


# Another one

In [1]:
import pandas as pd

# Load CSV
df = pd.read_csv("Cluster_n_sensitivity/WMA_fractions_v2_with_PV_GMM_ensemble.csv")

In [8]:
# Variables you want in Tecplot
variables = [
    "Longitude_[deg_E]",
    "Latitude_[deg_N]",
    "Depth_[m]",
    "Conservative_Temperature_[deg_C]",
    "Absolute_Salinity_[PSU]",
    "PV",
    "dPVdz",
    "year",
    "Dissolved_Oxygen_[micro_mol_per_kg]",
    "mean_cluster"
]

# Add any cluster probability variables
prob_cols = [c for c in df.columns if c.startswith("prob_mean_cluster_")]
variables += prob_cols
var_cols = [c for c in df.columns if c.startswith("var_per_sample_")]
variables += var_cols

In [10]:
# Keep only required columns
data = df[variables].dropna()

# Write Tecplot ASCII file
with open("arctic_water_masses.dat", "w") as f:

    f.write('TITLE = "Arctic Water Mass Classification"\n')

    f.write(
        "VARIABLES = " +
        " ".join(f'"{v}"' for v in variables) +
        "\n"
    )

    f.write(
        f'ZONE T="Water Mass Data", I={len(data)}, F=POINT\n'
    )

    # Write numerical data
    data.to_csv(
        f,
        sep=" ",
        index=False,
        header=False,
        float_format="%.8g"
    )